<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/5_01_silver_to_gold_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.01 From Bronze to Gold: the Card-Spend Challenge


## IMPORTANT NOTE!!
This whole thing is a stretch and an optional activity. The style is like the in-class test (experimentation), but the area is like the individual assessment (exploitation) ... i.e. you will not be asked to work like this in the test!!! The idea is that you can, if you want to, put together many of the things we have covered in this cards example as a final pipeline.

Consequently, engage with this how you wish. It's not critical preparation for either assessment, but could well be useful in building your learning of what an analytics engineering flow looks like. You can code from scratch, from the notebooks shared last week (`4.03` and `4.04`), you can use AI, you can use some combination of these.

In `4.01` you drew the DAG for the bank's silver layer. Now you build it. This notebook has four parts:

| Part | You will | How you know it works |
|---|---|---|
| A | Write the 16 silver transforms that were left as `pass` | Each function has a unit test with a tiny table |
| B | Put the 20 steps in the right order to build the silver table | Counts and checks on the real data |
| C | Decide what the three gold tables must contain, and write them | Columns, row counts and totals |
| D | Load silver and gold into DuckDB and answer seven business questions in SQL | Each answer is marked PASS or FAIL |

**The case, as a reminder.** The bank receives card transactions from a processor, weekly customer snapshots from its CRM and daily exchange rates. Management want spend in pounds by month and customer segment. The raw (bronze) data is messy, and the cleaning decisions are already agreed with the business. They are written into each function's description. Your job is to turn them into code.

**Resources.** `4.03` (pandas) and `4.04` (SQL) cover every function you need. Section numbers are given as hints.


## How this notebook works
- Each function cell has a description (what it does, what it *Needs*, what it *Adds*). Replace `pass` with code that does exactly that.
- Under each function is a **test cell**. Do not edit it. Run it after you write your function. It prints `PASS`, `FAIL` (with what it expected) or `ERROR` (your code crashed).
- The tests use tiny tables you can read in a few seconds, so you can see exactly what each function should do. The functions are independent: you can do them in any order.
- The rules for every function are the ones written in its description. Where the description records a decision (for example, "the month the purchase was MADE"), follow it even if you would have chosen differently.
- A function must **not change the table it is given**. Always work on a copy (`df.copy()`), as the examples do.
- When everything passes, use *Runtime > Restart session and run all* to check that it works from a clean start.

*You can use any resource, including AI tools, as you would at work. The tests will tell you whether it is right. Make sure you understand each function you submit, because Part D and the written questions need that.*

## Setup

In [ ]:
# ---- Setup: run this cell first ----
import glob, os, urllib.request, zipfile, shutil
import numpy as np
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dagsession/card_spend_dbt.zip"
DATA_DIR = "/content/card-spend"

# Force clean extraction every time to ensure integrity of files
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

zip_path = "/content/card_spend_dbt.zip"
urllib.request.urlretrieve(DATA_URL, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

def find(name):
    hits = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"Missing {name}. Checked {DATA_DIR}. Ask for help.")
    return hits[0]

bronze_tx = pd.read_csv(find("bronze_card_transactions.csv"), dtype={"ref_txn_id": "string"})
bronze_cu = pd.read_csv(find("bronze_customer_master.csv"))
bronze_fx = pd.read_csv(find("bronze_fx_rates.csv"))
print(f"card_transactions: {len(bronze_tx):,} rows | customer_master: {len(bronze_cu):,} rows | fx_rates: {len(bronze_fx):,} rows")

### The test helper
Run this cell too. It defines the little tools the test cells use. You do not need to read it.

In [ ]:
import hashlib

def need(result):
    """Stop a test with a clear message if the function still returns None."""
    assert result is not None, "the function returned None. Did you replace `pass` with code that ends in `return`?"
    return result

def run_test(test_fn):
    """Run one test function and print PASS, FAIL or ERROR."""
    name = test_fn.__name__
    name = name[len("test_"):] if name.startswith("test_") else name
    try:
        test_fn()
        print(f"PASS   {name}")
    except AssertionError as e:
        print(f"FAIL   {name}: {e}")
    except Exception as e:
        print(f"ERROR  {name}: {type(e).__name__}: {e}")

def _norm(x):
    try:
        return f"{float(x):.2f}"
    except (TypeError, ValueError):
        return str(x).strip().upper()

def _hash(x):
    return hashlib.sha256(_norm(x).encode()).hexdigest()[:12]

# Part A: The silver transforms

## Four are already written
These four were written for you in `4.01`. Read them: they show the style (a description, then code that works on a copy of the table). Run the cell.

In [ ]:
def drop_exact_duplicates(df):
    """Remove rows that are identical in every column.

    Why: the processor sometimes sends a whole daily file twice.
    Needs: the raw transactions, as delivered (before any columns are added)
    Adds:  nothing (fewer rows)
    """
    return df.drop_duplicates().reset_index(drop=True)


def standardise_currency_codes(df, col="currency"):
    """Trim spaces and upper-case currency codes ('gbp ' becomes 'GBP').

    Needs: currency
    Adds:  currency (cleaned, same column)
    """
    out = df.copy()
    out[col] = out[col].str.strip().str.upper()
    return out


def parse_timestamps(df):
    """Turn the text timestamp into a real datetime.
    Two formats arrive mixed: 'YYYY-MM-DD hh:mm:ss' and day-first 'dd/mm/yyyy hh:mm'.
    Anything with a '/' is read day first (an assumption worth stating).

    Needs: txn_timestamp
    Adds:  ts
    """
    out = df.copy()
    s = out["txn_timestamp"]
    day_first = s.str.contains("/", regex=False)
    ts = pd.Series(pd.NaT, index=out.index, dtype="datetime64[ns]")
    ts[day_first] = pd.to_datetime(s[day_first], format="%d/%m/%Y %H:%M")
    ts[~day_first] = pd.to_datetime(s[~day_first], format="%Y-%m-%d %H:%M:%S")
    out["ts"] = ts
    return out


def keep_latest_status_per_transaction(df):
    """Keep one row per transaction id: the latest known status.
    Latest = the most recent file date; if two rows share a file date, the higher
    status wins (PENDING < POSTED < REVERSED).

    Needs: txn_id, file_date, status
    Adds:  nothing (one row per txn_id)
    """
    out = df.copy()
    out["_rank"] = out["status"].map(STATUS_RANK)
    out = out.sort_values(["txn_id", "file_date", "_rank"])
    out = out.groupby("txn_id").tail(1).drop(columns="_rank")
    return out.reset_index(drop=True)


STATUS_RANK = {"PENDING": 0, "POSTED": 1, "REVERSED": 2}

In [ ]:
# Quick look at what the four given steps do to the raw transactions
demo = keep_latest_status_per_transaction(parse_timestamps(standardise_currency_codes(drop_exact_duplicates(bronze_tx))))
print(f"{len(bronze_tx):,} raw rows -> {len(demo):,} rows (one per transaction)")
demo[["txn_id", "currency", "status", "txn_timestamp", "ts"]].head(3)

## A1. Transactions
Steps that clean one table of card transactions.

### `standardise_merchant_text`
*Hint: `4.03` section 6 (text cleaning): strip, collapse repeated spaces with a regular expression, upper-case.*

In [ ]:
def standardise_merchant_text(df):
    """Tidy merchant names: trim, collapse repeated spaces, upper-case.
    The original text is kept in `merchant`.

    Needs: merchant
    Adds:  merchant_clean

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_standardise_merchant_text():
    df = pd.DataFrame({"merchant": ["  tesco   stores  4848 ", "Amazon Mktplace"]})
    before = df.copy()
    out = need(standardise_merchant_text(df))
    assert list(out["merchant_clean"]) == ["TESCO STORES 4848", "AMAZON MKTPLACE"], f"merchant_clean was {list(out['merchant_clean'])}"
    assert list(out["merchant"]) == list(before["merchant"]), "the original `merchant` column must be kept as it was"
    assert df.equals(before), "the function changed the table it was given: work on a copy"
run_test(test_standardise_merchant_text)

### `derive_transaction_date_and_month`
*Hint: `4.03` section 8 (dates): the `.dt` accessor, `normalize()` and `strftime`.*

In [ ]:
def derive_transaction_date_and_month(df):
    """Work out the transaction date and its month (YYYY-MM).
    Decision recorded here: the month is the month the purchase was MADE, not the month
    the file arrived.

    Needs: ts
    Adds:  tdate, month

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_derive_transaction_date_and_month():
    df = pd.DataFrame({"ts": pd.to_datetime(["2026-01-31 23:59:00", "2026-02-01 00:01:00"])})
    out = need(derive_transaction_date_and_month(df))
    assert list(out["month"]) == ["2026-01", "2026-02"], f"month was {list(out['month'])}"
    assert list(out["tdate"]) == list(pd.to_datetime(["2026-01-31", "2026-02-01"])), f"tdate was {list(out['tdate'])} (the date with the time set to midnight)"
    assert "ts" in out.columns, "keep the original ts column"
run_test(test_derive_transaction_date_and_month)

### `flag_reversals`
*Hint: A reversal is a row whose status is `REVERSED`.*

In [ ]:
def flag_reversals(df):
    """Mark rows that are reversals (a refund recorded as its own row).

    Needs: status
    Adds:  is_reversal (True/False)

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_flag_reversals():
    df = pd.DataFrame({"status": ["POSTED", "REVERSED", "PENDING"]})
    out = need(flag_reversals(df))
    assert list(out["is_reversal"]) == [False, True, False], f"is_reversal was {list(out['is_reversal'])}"
    assert "status" in out.columns
run_test(test_flag_reversals)

### `flag_zero_amount`
*Hint: Compare the amount to zero. Keep the rows; just flag them.*

In [ ]:
def flag_zero_amount(df):
    """Mark transactions with an amount of exactly zero (keep them, flagged).

    Needs: amount
    Adds:  zero_amount (True/False)

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_flag_zero_amount():
    df = pd.DataFrame({"amount": [0.0, 10.0, -5.0, 0.0]})
    out = need(flag_zero_amount(df))
    assert list(out["zero_amount"]) == [True, False, False, True], f"zero_amount was {list(out['zero_amount'])}"
    assert len(out) == 4, "no rows may be dropped"
run_test(test_flag_zero_amount)

### `link_reversals_to_original`
*Hint: `4.03` section 13 (combining): build a small table of the originals (`txn_id` and `amount`), then a left merge on `ref_txn_id`. Check the row count afterwards.*

In [ ]:
def link_reversals_to_original(df):
    """Connect each reversal row to the original purchase it cancels,
    and flag reversals whose original cannot be found.

    Needs: txn_id, ref_txn_id, is_reversal (one row per txn_id, so originals can be found)
    Adds:  original_found  True only for a reversal whose original purchase is in the table
                           (False for everything else)
           original_amount the amount of that original purchase (missing for every other row)
    The number of rows and their order must not change.

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_link_reversals_to_original():
    df = pd.DataFrame({
        "txn_id": ["T1", "T2", "T3", "T4"],
        "ref_txn_id": [pd.NA, "T1", pd.NA, "T9"],      # T2 reverses T1; T4 points at T9, which is not in the table
        "amount": [50.0, -50.0, 20.0, -30.0],
        "is_reversal": [False, True, False, True],
    })
    out = need(link_reversals_to_original(df))
    assert len(out) == 4, f"the table has {len(out)} rows; it must still have 4 (a merge can multiply rows)"
    assert list(out["txn_id"]) == ["T1", "T2", "T3", "T4"], "the row order must not change"
    assert list(out["original_found"]) == [False, True, False, False], f"original_found was {list(out['original_found'])}"
    assert out.loc[out["txn_id"] == "T2", "original_amount"].iloc[0] == 50.0, "T2 reverses a purchase of 50.0"
    assert out["original_amount"].isna().sum() == 3, "original_amount should be missing for T1, T3 (not reversals) and T4 (original not found)"
run_test(test_link_reversals_to_original)

### `flag_foreign_currency`
*Hint: Compare the (already standardised) currency to `"GBP"`.*

In [ ]:
def flag_foreign_currency(df):
    """Mark transactions made in a currency other than GBP.

    Needs: currency
    Adds:  is_foreign_currency (True/False)

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_flag_foreign_currency():
    df = pd.DataFrame({"currency": ["GBP", "EUR", "USD", "GBP"]})
    out = need(flag_foreign_currency(df))
    assert list(out["is_foreign_currency"]) == [False, True, True, False], f"is_foreign_currency was {list(out['is_foreign_currency'])}"
run_test(test_flag_foreign_currency)

## A2. Customers
Steps that clean the weekly customer snapshots.

### `standardise_segment_labels`
*Hint: `4.03` section 6: strip spaces and use title case (`str.title()`).*

In [ ]:
def standardise_segment_labels(df):
    """Make segment labels consistent ('premium', 'PREMIUM ' become 'Premium').

    Needs: segment
    Adds:  segment (cleaned, same column)

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_standardise_segment_labels():
    df = pd.DataFrame({"segment": [" premium", "STANDARD ", "Premium", "standard"]})
    out = need(standardise_segment_labels(df))
    assert list(out["segment"]) == ["Premium", "Standard", "Premium", "Standard"], f"segment was {list(out['segment'])}"
run_test(test_standardise_segment_labels)

### `latest_customer_snapshot_per_card`
*Hint: `4.03` sections 5 and 9: sort by the snapshot date, then keep the last row per card. Convert the dates first so they sort as dates.*

In [ ]:
def latest_customer_snapshot_per_card(df):
    """Reduce the weekly snapshots to one row per card: the most recent record.
    Decision recorded here: 'segment' means the segment NOW (latest snapshot).

    Needs: card_id, snapshot_date, customer_id, segment, account_status
    Adds:  nothing (one row per card)

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_latest_customer_snapshot_per_card():
    df = pd.DataFrame({
        "snapshot_date": ["2026-01-12", "2026-01-05", "2026-01-12", "2026-01-05"],   # deliberately not in date order
        "customer_id":   ["CU1", "CU1", "CU2", "CU2"],
        "account_id":    ["A1", "A1", "A2", "A2"],
        "card_id":       ["C1", "C1", "C2", "C2"],
        "segment":       ["Premium", "Standard", "Standard", "Premium"],
        "account_status": ["ACTIVE", "ACTIVE", "CLOSED", "ACTIVE"],
        "open_date":     ["2020-01-01"] * 4,
    })
    out = need(latest_customer_snapshot_per_card(df))
    assert list(out.columns) == ["card_id", "customer_id", "segment", "account_status"], f"columns were {list(out.columns)}"
    out = out.sort_values("card_id").reset_index(drop=True)
    assert len(out) == 2, f"expected one row per card (2) but got {len(out)}"
    assert list(out["segment"]) == ["Premium", "Standard"], f"segment was {list(out['segment'])}: it must come from the LATEST snapshot"
    assert list(out["account_status"]) == ["ACTIVE", "CLOSED"], f"account_status was {list(out['account_status'])}"
run_test(test_latest_customer_snapshot_per_card)

## A3. Exchange rates
Steps that turn the daily rates into a table with a rate for every day, including weekends.

### `build_fx_calendar`
*Hint: `4.03` section 8: `pd.date_range` for the days, `pivot` to one column per currency, `reindex` onto the full calendar.*

In [ ]:
def build_fx_calendar(fx, end_date):
    """Lay the rates out on a full calendar: one row per day from the first
    published rate to `end_date`, one column per currency. Days with no published rate
    (weekends, holidays) are left empty at this stage.

    Needs: rate_date, currency, rate_to_gbp
    Adds:  a table indexed by day, one column per currency

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_build_fx_calendar():
    fx = pd.DataFrame({
        "rate_date": ["2026-01-01", "2026-01-01", "2026-01-03", "2026-01-03"],
        "currency": ["EUR", "USD", "EUR", "USD"],
        "rate_to_gbp": [0.86, 0.79, 0.87, 0.80],
    })
    out = need(build_fx_calendar(fx, "2026-01-04"))
    assert list(out.index) == list(pd.date_range("2026-01-01", "2026-01-04")), f"the index should be every day from 2026-01-01 to 2026-01-04, but was {list(out.index)}"
    assert set(out.columns) == {"EUR", "USD"}, f"expected one column per currency, got {list(out.columns)}"
    assert pd.isna(out.loc["2026-01-02", "EUR"]), "a day with no published rate must be left empty at this stage"
    assert out.loc["2026-01-03", "USD"] == 0.80
run_test(test_build_fx_calendar)

### `fill_missing_fx_rates`
*Hint: `4.03` section 4 (missing values): carry the last known value forward.*

In [ ]:
def fill_missing_fx_rates(fx_wide):
    """Fill days with no published rate with the last published rate.
    Decision recorded here: weekends and holidays use the previous published day.

    Needs: the calendar table from build_fx_calendar
    Adds:  nothing (empty days are filled)

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_fill_missing_fx_rates():
    wide = pd.DataFrame({"EUR": [0.86, np.nan, np.nan, 0.87]}, index=pd.date_range("2026-01-01", periods=4))
    before = wide.copy()
    out = need(fill_missing_fx_rates(wide))
    assert list(out["EUR"]) == [0.86, 0.86, 0.86, 0.87], f"EUR was {list(out['EUR'])}"
    assert wide.equals(before) or wide.isna().equals(before.isna()), "the function changed the table it was given"
run_test(test_fill_missing_fx_rates)

### `add_gbp_rate`
*Hint: A pound is worth one pound.*

In [ ]:
def add_gbp_rate(fx_wide):
    """Add GBP itself with a rate of 1.0, so GBP transactions convert like any other.

    Needs: the calendar table
    Adds:  a GBP column (all 1.0)

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_add_gbp_rate():
    wide = pd.DataFrame({"EUR": [0.86, 0.87]}, index=pd.date_range("2026-01-01", periods=2))
    out = need(add_gbp_rate(wide))
    assert list(out["GBP"]) == [1.0, 1.0], f"GBP was {list(out['GBP'])}"
    assert list(out["EUR"]) == [0.86, 0.87], "the other columns must be unchanged"
    assert "GBP" not in wide.columns, "the function changed the table it was given: work on a copy"
run_test(test_add_gbp_rate)

### `flag_filled_fx_rates`
*Hint: This one runs on the calendar table *before* it is filled (so it is independent of `fill_missing_fx_rates`). A day is flagged if any currency has no rate that day.*

In [ ]:
def flag_filled_fx_rates(fx_wide):
    """Record which days used a carried-forward rate instead of a published
    one, so a user can tell a real rate from a filled one.

    Needs: the calendar table BEFORE it is filled
    Returns: a True/False Series named rate_is_filled, with the same day index as the
             calendar table. True on a day when a rate was missing (filled later).

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_flag_filled_fx_rates():
    wide = pd.DataFrame({"EUR": [0.86, np.nan, 0.87, 0.88], "USD": [0.79, np.nan, np.nan, 0.80]}, index=pd.date_range("2026-01-01", periods=4))
    out = need(flag_filled_fx_rates(wide))
    assert list(out) == [False, True, True, False], f"expected [False, True, True, False] but got {list(out)} (a day counts if ANY currency has no rate)"
    assert list(out.index) == list(wide.index), "the result must have the same day index as the calendar table"
run_test(test_flag_filled_fx_rates)

## A4. Joining and enriching
Steps that bring the pieces together.

### `add_segment_from_latest_snapshot`
*Hint: `4.03` section 13: a left merge, then `fillna`. Cards missing from the customer table must be kept, not dropped.*

In [ ]:
def add_segment_from_latest_snapshot(tx, cards):
    """Attach each transaction's customer segment (latest snapshot).
    Cards missing from the customer master get segment 'Unknown' and are flagged,
    not dropped, so totals still reconcile.

    Needs: tx: card_id | cards: card_id, segment (one row per card)
    Adds:  segment, customer_known (True/False)

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_add_segment_from_latest_snapshot():
    tx = pd.DataFrame({"txn_id": ["T1", "T2", "T3"], "card_id": ["C1", "C2", "C9"]})
    cards = pd.DataFrame({"card_id": ["C1", "C2"], "segment": ["Premium", "Standard"]})
    out = need(add_segment_from_latest_snapshot(tx, cards))
    assert len(out) == 3, f"expected 3 rows but got {len(out)}: a card missing from the customer table must not drop its transaction"
    assert list(out["segment"]) == ["Premium", "Standard", "Unknown"], f"segment was {list(out['segment'])}"
    assert list(out["customer_known"]) == [True, True, False], f"customer_known was {list(out['customer_known'])}"
run_test(test_add_segment_from_latest_snapshot)

### `add_fx_rate_to_transactions`
*Hint: Reshape the calendar table to long form (`melt`), then merge on date **and** currency. Rename the index first so the date column has a name you can merge on.*

In [ ]:
def add_fx_rate_to_transactions(tx, fx_wide):
    """Look up the exchange rate for each transaction: the rate on the
    transaction DATE, for the transaction's currency.

    Needs: tx: tdate, currency | fx_wide: the filled calendar table including GBP
    Adds:  rate

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_add_fx_rate_to_transactions():
    tx = pd.DataFrame({
        "txn_id": ["T1", "T2", "T3"],
        "tdate": pd.to_datetime(["2026-01-01", "2026-01-02", "2026-01-02"]),
        "currency": ["EUR", "GBP", "EUR"],
    })
    fx_wide = pd.DataFrame({"EUR": [0.86, 0.87], "GBP": [1.0, 1.0]}, index=pd.date_range("2026-01-01", periods=2))
    out = need(add_fx_rate_to_transactions(tx, fx_wide))
    assert len(out) == 3, f"expected 3 rows but got {len(out)}"
    assert list(out["rate"]) == [0.86, 1.0, 0.87], f"rate was {list(out['rate'])}"
run_test(test_add_fx_rate_to_transactions)

### `add_gbp_amount`
*Hint: Multiply the amount by the rate.*

In [ ]:
def add_gbp_amount(df):
    """Convert each amount to pounds.

    Needs: amount, rate
    Adds:  amount_gbp

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_add_gbp_amount():
    df = pd.DataFrame({"amount": [100.0, 50.0], "rate": [0.5, 1.0]})
    out = need(add_gbp_amount(df))
    assert list(out["amount_gbp"]) == [50.0, 50.0], f"amount_gbp was {list(out['amount_gbp'])}"
run_test(test_add_gbp_amount)

## A5. The check

### `check_one_row_per_transaction`
*Hint: Return the table if it is fine; `raise ValueError("...")` if not. This is the same rule as dbt's `unique` and `not_null` tests.*

In [ ]:
def check_one_row_per_transaction(df):
    """Check that txn_id is unique and never empty in the finished silver table.

    Needs: txn_id, in the finished table
    Raises: ValueError if any txn_id is missing or repeated
    Returns: the table unchanged, so the check can sit at the end of a chain of steps

    """
    # YOUR CODE HERE
    pass

In [ ]:
def test_check_one_row_per_transaction():
    good = pd.DataFrame({"txn_id": ["T1", "T2"]})
    assert need(check_one_row_per_transaction(good)).equals(good), "a good table must be returned unchanged"
    for label, bad in [("a repeated id", pd.DataFrame({"txn_id": ["T1", "T1"]})),
                       ("a missing id", pd.DataFrame({"txn_id": ["T1", None]}))]:
        try:
            check_one_row_per_transaction(bad)
        except ValueError:
            continue
        raise AssertionError(f"{label} should raise a ValueError, but nothing was raised")
run_test(test_check_one_row_per_transaction)

# Part B: Build the silver table

All 20 steps now exist. Put them in a valid order. **Use the DAG you drew in `4.01`**: three branches (transactions, customers, exchange rates) that meet at the three joins.

Write `build_silver`, which takes the three bronze tables and returns one silver table with one row per transaction.

Things to remember:
- The cleaning decisions are in the function descriptions. Your job here is the *order*, driven by each step's *Needs* and *Adds*.
- Some steps change the number of rows (duplicates, latest status), and the joins must not.
- There is not one answer to this question!
- Run the check last.

In [ ]:
def build_silver(bronze_tx, bronze_cu, bronze_fx, end_date="2026-03-31"):
    """Run the steps in a valid order and return the silver transactions table.

    Needs: the three bronze tables
    Returns: one row per transaction, with these columns added to the raw ones:
             merchant_clean, ts, tdate, month, is_reversal, zero_amount, original_amount,
             original_found, is_foreign_currency, segment, customer_known, rate, amount_gbp
    """
    # YOUR CODE HERE
    pass

In [ ]:
silver = build_silver(bronze_tx, bronze_cu, bronze_fx)
silver.head(3) if silver is not None else print("build_silver returned None")

### Tests on the real data

In [ ]:
def _silver():
    return need(silver)

def test_silver_one_row_per_transaction():
    s = _silver()
    assert len(s) == 15126, f"expected 15,126 rows (one per transaction) but got {len(s):,}"
    assert s["txn_id"].is_unique, "txn_id is not unique"

def test_silver_has_the_expected_columns():
    s = _silver()
    need_cols = ["txn_id", "card_id", "amount", "currency", "status", "merchant_clean", "ts", "tdate", "month",
                 "is_reversal", "zero_amount", "original_found", "original_amount", "is_foreign_currency",
                 "segment", "customer_known", "rate", "amount_gbp"]
    missing = [c for c in need_cols if c not in s.columns]
    assert not missing, f"missing columns: {missing}"

def test_silver_every_transaction_has_a_gbp_amount():
    s = _silver()
    n = s["amount_gbp"].isna().sum()
    assert n == 0, f"{n} transactions have no GBP amount. Check the dates, the currency codes and the GBP rate."

def test_silver_unknown_cards_are_kept():
    s = _silver()
    n = (~s["customer_known"]).sum()
    assert n == 150, f"expected 150 transactions on cards missing from the customer master but got {n}"
    assert set(s["segment"].unique()) == {"Premium", "Standard", "Unknown"}, f"segments were {sorted(s['segment'].unique())}"

def test_silver_flags():
    s = _silver()
    assert s["is_reversal"].sum() == 281, f"expected 281 reversals but got {s['is_reversal'].sum()}"
    assert s["zero_amount"].sum() == 59, f"expected 59 zero-amount rows but got {s['zero_amount'].sum()}"
    assert s["is_foreign_currency"].sum() == 1387, f"expected 1,387 foreign-currency rows but got {s['is_foreign_currency'].sum()}"
    assert s["original_found"].sum() == 281, f"expected all 281 reversals to find their original but got {s['original_found'].sum()}"

def test_silver_reversals_cancel_their_originals():
    s = _silver()
    r = s[s["is_reversal"]]
    assert ((r["amount"] + r["original_amount"]).abs() < 1e-9).all(), "a reversal's amount should be the negative of the original purchase's amount"

def test_bronze_is_untouched():
    assert len(bronze_tx) == 28829 and "ts" not in bronze_tx.columns, "build_silver changed the bronze table. Work on copies."

for t in [test_silver_one_row_per_transaction, test_silver_has_the_expected_columns,
          test_silver_every_transaction_has_a_gbp_amount, test_silver_unknown_cards_are_kept,
          test_silver_flags, test_silver_reversals_cancel_their_originals, test_bronze_is_untouched]:
    run_test(t)

# Part C: Gold

Silver holds one clean row per transaction. Gold answers the business's questions. Management have asked for three tables.

| Table | The question | Columns (exactly these, in this order) |
|---|---|---|
| `gold_spend_by_month_segment` | How much did customers spend, in pounds, in each month and customer segment? | `month`, `segment`, `spend_gbp`, `transactions` |
| `gold_reversals_by_month` | How many reversals were there each month, and what were they worth? | `month`, `reversals`, `value_gbp` |
| `gold_foreign_currency_by_month` | How much was spent in a currency other than sterling each month, and by how many cards? | `month`, `cards`, `transactions`, `spend_gbp` |

**Rules agreed with the business**
- *Spend* means transactions that have **posted**. A pending transaction is not spend yet.
- Reversals are reported **separately**. They are not netted off the spend figures.
- Report reversal value as a **positive** number of pounds.
- Every amount is in GBP (`amount_gbp`), rounded to 2 decimal places. `cards` is the number of *different* cards.
- Cards missing from the customer master stay in the figures under the segment `Unknown`.
- Sort by `month`, then `segment` where there is one.

Decide what to filter and how to group, then write one function per table. Each takes the silver table and returns a DataFrame. (`4.03` section 10.)

### `gold_spend_by_month_segment`
*Hint: Filter to posted rows first, then `groupby(['month', 'segment'])` with named aggregation.*

In [ ]:
def gold_spend_by_month_segment(silver):
    """One row per month and segment: posted spend in GBP and the number of posted transactions."""
    # YOUR CODE HERE
    pass

In [ ]:
def test_gold_spend_by_month_segment():
    g = need(gold_spend_by_month_segment(silver))
    assert list(g.columns) == ["month", "segment", "spend_gbp", "transactions"], f"columns were {list(g.columns)}"
    assert len(g) == 9, f"expected 9 rows (3 months x 3 segments) but got {len(g)}"
    assert abs(g["spend_gbp"].sum() - 450715.19) < 0.02, f"total spend_gbp was {g['spend_gbp'].sum():,.2f}; expected 450,715.19"
    assert g["transactions"].sum() == 14388, f"total transactions was {g['transactions'].sum():,}; expected 14,388"
    assert list(g["month"]) == sorted(g["month"]), "the table must be sorted by month, then segment"
    assert g["spend_gbp"].round(2).equals(g["spend_gbp"]), "spend_gbp must be rounded to 2 decimal places"
run_test(test_gold_spend_by_month_segment)

### `gold_reversals_by_month`
*Hint: Reversal rows are already negative amounts. Which sign do you report?*

In [ ]:
def gold_reversals_by_month(silver):
    """One row per month: the number of reversals and their value in GBP (positive)."""
    # YOUR CODE HERE
    pass

In [ ]:
def test_gold_reversals_by_month():
    g = need(gold_reversals_by_month(silver))
    assert list(g.columns) == ["month", "reversals", "value_gbp"], f"columns were {list(g.columns)}"
    assert len(g) == 3, f"expected 3 rows (one per month) but got {len(g)}"
    assert g["reversals"].sum() == 281, f"total reversals was {g['reversals'].sum()}; expected 281"
    assert abs(g["value_gbp"].sum() - 8104.23) < 0.02, f"total value_gbp was {g['value_gbp'].sum():,.2f}; expected 8,104.23 (positive pounds)"
run_test(test_gold_reversals_by_month)

### `gold_foreign_currency_by_month`
*Hint: `nunique` counts distinct values inside a group.*

In [ ]:
def gold_foreign_currency_by_month(silver):
    """One row per month: posted spend in a foreign currency, with the number of transactions and distinct cards."""
    # YOUR CODE HERE
    pass

In [ ]:
def test_gold_foreign_currency_by_month():
    g = need(gold_foreign_currency_by_month(silver))
    assert list(g.columns) == ["month", "cards", "transactions", "spend_gbp"], f"columns were {list(g.columns)}"
    assert len(g) == 3, f"expected 3 rows (one per month) but got {len(g)}"
    assert g["transactions"].sum() == 1321, f"total transactions was {g['transactions'].sum():,}; expected 1,321"
    assert abs(g["spend_gbp"].sum() - 98995.59) < 0.02, f"total spend_gbp was {g['spend_gbp'].sum():,.2f}; expected 98,995.59"
    assert g["cards"].sum() == 819, f"cards (distinct per month) summed to {g['cards'].sum()}; expected 819"
run_test(test_gold_foreign_currency_by_month)

In [ ]:
gold_spend = gold_spend_by_month_segment(silver)
gold_reversals = gold_reversals_by_month(silver)
gold_foreign = gold_foreign_currency_by_month(silver)
gold_spend

# Part D: Query it in DuckDB

Now the warehouse side. Load the silver table and the three gold tables into DuckDB, then answer seven questions from management in SQL (`4.04` has every pattern you need).

The principle: **headline numbers come from gold, detail comes from silver.** Questions 1 and 7 use the gold tables; the others need row-level detail from silver.

In [ ]:
%pip install -q duckdb
import duckdb

for _name, _obj in [("silver", silver), ("gold_spend", gold_spend), ("gold_reversals", gold_reversals), ("gold_foreign", gold_foreign)]:
    assert _obj is not None, f"{_name} has not been built yet: finish Parts B and C first"

con = duckdb.connect()
con.register("silver_df", silver)
con.register("gold_spend_df", gold_spend)
con.register("gold_reversals_df", gold_reversals)
con.register("gold_foreign_df", gold_foreign)
con.sql("CREATE OR REPLACE TABLE silver AS SELECT * FROM silver_df")
con.sql("CREATE OR REPLACE TABLE gold_spend_by_month_segment AS SELECT * FROM gold_spend_df")
con.sql("CREATE OR REPLACE TABLE gold_reversals_by_month AS SELECT * FROM gold_reversals_df")
con.sql("CREATE OR REPLACE TABLE gold_foreign_currency_by_month AS SELECT * FROM gold_foreign_df")

def q(sql):
    """Run SQL and return the result as a table."""
    result = con.sql(sql)
    return None if result is None else result.df()

q("SELECT table_name, estimated_size AS rows FROM duckdb_tables() ORDER BY table_name")

In [ ]:
# Look around first. What does silver look like to SQL?
q("DESCRIBE silver")

In [ ]:
# The marking helper: it compares your answer to the expected one without showing it.
ANSWERS = {
    "S1": "e10d8b15d361", "Q1": "5fc039a58195", "Q2": "35e594c4c746", "Q3": "23759997b3c5", "Q4": "af9e96e49814", "Q5": "2123ab0bd9bc", "Q6": "7734a6cc1a11", "Q7": "80bb76ee49e4",
}

def check_answer(label, value):
    ok = _hash(value) == ANSWERS[label]
    print(("PASS   " if ok else "FAIL   ") + f"{label}: you answered {_norm(value)}" + ("" if ok else ". Check the filter, the rounding and the column you used."))

### Q1
What was the total posted spend, in pounds, for **Premium** customers in **March 2026**? (2 decimal places; use a gold table.)

In [ ]:
answer_q1 = q("""
SELECT NULL AS answer   -- replace this line with your query
""").iloc[0, 0]
check_answer("Q1", answer_q1)

### Q2
What percentage of all posted spend (GBP) was in a foreign currency? (1 decimal place, e.g. `12.3`.)

In [ ]:
answer_q2 = q("""
SELECT NULL AS answer   -- replace this line with your query
""").iloc[0, 0]
check_answer("Q2", answer_q2)

### Q3
Which segment (`Premium`, `Standard` or `Unknown`) has the highest **average** posted transaction value in GBP?

In [ ]:
answer_q3 = q("""
SELECT NULL AS answer   -- replace this line with your query
""").iloc[0, 0]
check_answer("Q3", answer_q3)

### Q4
On which day of the week (for example `Monday`) is total posted spend in GBP highest? Use the purchase time `ts`.

In [ ]:
answer_q4 = q("""
SELECT NULL AS answer   -- replace this line with your query
""").iloc[0, 0]
check_answer("Q4", answer_q4)

### Q5
How much posted spend (GBP, 2 decimal places) is on cards that are **not** in the customer master?

In [ ]:
answer_q5 = q("""
SELECT NULL AS answer   -- replace this line with your query
""").iloc[0, 0]
check_answer("Q5", answer_q5)

### Q6
Which `card_id` has the highest total posted spend in GBP?

In [ ]:
answer_q6 = q("""
SELECT NULL AS answer   -- replace this line with your query
""").iloc[0, 0]
check_answer("Q6", answer_q6)

### Q7
In which month were reversals highest **as a percentage of posted transactions**? Answer with the month, e.g. `2026-01`. (Use both gold tables.)

In [ ]:
answer_q7 = q("""
SELECT NULL AS answer   -- replace this line with your query
""").iloc[0, 0]
check_answer("Q7", answer_q7)

### Before you move on
Look at your seven queries. Which ones could have been answered from gold, and which could not? What would have to change in the gold tables for the others to be answered from gold too? Keep that thought for the written questions.

---
### Finished?
Use *Runtime > Restart session and run all*. Every test should say `PASS`. If a cell shows `FAIL` or `ERROR`, read the message: it says what it expected.